<a href="https://colab.research.google.com/github/JianfengMI/MLprojects/blob/main/Engulfing_EP_Momentum_Signals_Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
############################################################################
# Helper files for Engulfing Bars System
############################################################################
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import math
from tqdm.auto import tqdm
import requests
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# download S&P500 tickers and their prices
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    # pandas can directly parse tables from HTML content
    tables = pd.read_html(response.text)

    df = None
    # Iterate through found tables to identify the S%P 500 constituents table
    for df_candidate in tables:
        # Check for columns commonly present in the S%P 500 constituents table
        # like 'Symbol', 'Ticker', or 'symbol' (case-insensitive)
        # Convert column names to string before comparison
        cols = [c for c in df_candidate.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()]
        if cols:
            df = df_candidate
            break # Found the table

    if df is None:
        raise ValueError("Could not find S%26P 500 constituents table on the Wikipedia page.")

    # Ensure 'Symbol' or 'Ticker' column is correctly identified
    # Convert column names to string before comparison
    col_name = [c for c in df.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()][0]

    # Clean up ticker symbols (e.g., BRK.B -> BRK-B)
    tickers = df[col_name].astype(str).str.replace(".", "-", regex=False).tolist()
    return tickers

def download_stock_prices(tickers_to_download, start, end, batch_size=80):
    """
    Download full OHLCV price data in batches using yfinance.
    Returns a MultiIndex DataFrame:
        columns = (ticker, price_field)
        index   = dates
    """
    all_data = pd.DataFrame()
    tickers_list = list(tickers_to_download)

    for i in tqdm(range(0, len(tickers_list), batch_size), desc="Downloading prices"):
        batch = tickers_list[i:i+batch_size]

        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker', auto_adjust=True
            )

            if data.empty:
                print(f"Skipping batch {batch}: No data downloaded.")
                continue

            # If single ticker, YF returns normal DataFrame → convert it to MultiIndex
            if not isinstance(data.columns, pd.MultiIndex):
                t = batch[0]
                data.columns = pd.MultiIndex.from_product([[t], data.columns])

            # Merge batch with accumulated data
            if all_data.empty:
                all_data = data
            else:
                all_data = pd.concat([all_data, data], axis=1)

        except Exception as e:
            print(f"Batch download error for {batch}: {e}")

    # Ensure sorted index and columns
    if not all_data.empty:
        all_data = all_data.sort_index()
        all_data = all_data.sort_index(axis=1)

    return all_data

# mount on google drive and save the data
from google.colab import drive
drive.mount('/content/drive')

def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

def calculate_atr(df, period=14):
    high = df['High']
    low = df['Low']
    close = df['Close']

    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    return atr


def calculate_trend_slope(series, window=20):
    return (series - series.shift(window)) / window

def calculate_rsi(data, period=14):
    delta = data.diff()

    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)

    avg_gain = gains.ewm(com=period-1, min_periods=period).mean()
    avg_loss = losses.ewm(com=period-1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def screen_engulfing_signals(df, min_ma_period=50):
    all_signals = []

    tickers = df.columns.get_level_values(0).unique()

    for ticker in tickers:
        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'regime']
        if not all((ticker, c) in df.columns for c in required_cols):
            continue

        price_data = df[ticker].dropna().copy()

        if len(price_data) < 200:  # ensure enough for EMA200 etc
            continue

        # =========================
        # Core Indicators
        # =========================
        price_data['MA10'] = price_data['Close'].rolling(10).mean()
        price_data['MA20'] = price_data['Close'].rolling(20).mean()
        price_data['MA50'] = price_data['Close'].rolling(50).mean()
        price_data['EMA200'] = price_data['Close'].ewm(span=200, adjust=False).mean()

        price_data['EMA12'] = price_data['Close'].ewm(span=12, adjust=False).mean()
        price_data['EMA26'] = price_data['Close'].ewm(span=26, adjust=False).mean()
        price_data['MACD'] = price_data['EMA12'] - price_data['EMA26']
        price_data['Signal_Line'] = price_data['MACD'].ewm(span=9).mean()
        price_data['MACD_Histogram'] = price_data['MACD'] - price_data['Signal_Line']

        price_data['RSI'] = calculate_rsi(price_data['Close'])

        # =========================
        # NEW FEATURES
        # =========================

        # Volatility
        price_data['ATR'] = calculate_atr(price_data)

        # Trend
        price_data['Trend_Slope'] = calculate_trend_slope(price_data['MA50'], 20)
        price_data['Trend_Strength'] = price_data['Trend_Slope'] / price_data['Close']

        # Volume
        price_data['Vol_Avg20'] = price_data['Volume'].rolling(20).mean()
        price_data['Volume_Ratio'] = price_data['Volume'] / price_data['Vol_Avg20']

        # Key Levels
        price_data['Recent_High'] = price_data['High'].rolling(50).max()
        price_data['Recent_Low'] = price_data['Low'].rolling(50).min()

        price_data['Dist_EMA200'] = (price_data['Close'] - price_data['EMA200']) / price_data['EMA200']
        price_data['Dist_Recent_High'] = (price_data['Close'] - price_data['Recent_High']) / price_data['Recent_High']
        price_data['Dist_Recent_Low'] = (price_data['Close'] - price_data['Recent_Low']) / price_data['Recent_Low']

        # =========================
        # Convert to numpy
        # =========================
        o = price_data['Open'].values
        h = price_data['High'].values
        l = price_data['Low'].values
        c = price_data['Close'].values
        atr = price_data['ATR'].values

        for i in range(1, len(price_data)):
            if i < min_ma_period:
                continue

            # Engulfing condition
            if not (h[i] >= h[i-1] and l[i] <= l[i-1]):
                continue

            prev_red = c[i-1] < o[i-1]
            prev_green = c[i-1] >= o[i-1]
            cur_green = c[i] >= o[i]
            cur_red = c[i] < o[i]

            action = 0
            if prev_red and cur_green:
                action = 1
            elif prev_green and cur_red:
                action = -1
            elif prev_red and cur_red:
                action = 0.5
            elif prev_green and cur_green:
                action = -0.5

            if action == 0:
                continue

            # Volatility filter feature
            bar_range = h[i] - l[i]
            range_vs_atr = bar_range / atr[i] if atr[i] > 0 else np.nan

            all_signals.append({
                "Ticker": ticker,
                "Signal_Date": price_data.index[i],
                "Action": action,
                "Entry": c[i],
                "Stop": l[i] if action == 1 else h[i],

                # Core
                "MA10": price_data['MA10'].iloc[i],
                "MA20": price_data['MA20'].iloc[i],
                "MA50": price_data['MA50'].iloc[i],
                "RSI": price_data['RSI'].iloc[i],
                "MACD": price_data['MACD'].iloc[i],
                "Signal_Line": price_data['Signal_Line'].iloc[i],
                "MACD_Histogram": price_data['MACD_Histogram'].iloc[i],
                "regime": price_data['regime'].iloc[i],

                # NEW FEATURES
                "ATR": price_data['ATR'].iloc[i],
                "Range_vs_ATR": range_vs_atr,
                "Trend_Slope": price_data['Trend_Slope'].iloc[i],
                "Trend_Strength": price_data['Trend_Strength'].iloc[i],
                "Volume_Ratio": price_data['Volume_Ratio'].iloc[i],
                "Dist_EMA200": price_data['Dist_EMA200'].iloc[i],
                "Dist_Recent_High": price_data['Dist_Recent_High'].iloc[i],
                "Dist_Recent_Low": price_data['Dist_Recent_Low'].iloc[i]
            })

    if not all_signals:
        return pd.DataFrame()

    return pd.DataFrame(all_signals).sort_values("Signal_Date", ascending=False)

# revised signal scoring system
def compute_signal_score(row, weights=None):
    if weights is None:
        weights = {
            'vol_exp': 25,
            'vol_confirm': 25,
            'trend_align': 20,
            'overext_penalty': 15,
            'location': 15
        }

    score = 0.0

    # 1. Volatility Expansion (Range vs ATR)
    range_atr = row.get("Range_vs_ATR", 0)
    if range_atr > 2.0:
        score += weights['vol_exp']
    elif range_atr > 1.3:
        score += weights['vol_exp'] * 0.6
    elif range_atr > 1.0:
        score += weights['vol_exp'] * 0.3

    # 2. Volume Confirmation
    vol_ratio = row.get("Volume_Ratio", 0)
    if vol_ratio > 2.0:
        score += weights['vol_confirm']
    elif vol_ratio > 1.5:
        score += weights['vol_confirm'] * 0.7
    elif vol_ratio > 1.2:
        score += weights['vol_confirm'] * 0.4

    # 3. Trend Alignment
    action = row.get("Action", 0)
    trend_slope = row.get("Trend_Slope", 0)
    if action == 1 and trend_slope > 0:
        score += weights['trend_align']
    elif action == -1 and trend_slope < 0:
        score += weights['trend_align']
    else:
        score -= weights['trend_align'] * 0.6  # softer penalty

    # 4. Avoid Overextension
    dist_ema = abs(row.get("Dist_EMA200", 0))
    if dist_ema > 0.20:
        score -= weights['overext_penalty']
    elif dist_ema > 0.12:
        score -= weights['overext_penalty'] * 0.6

    # 5. Location Advantage (closer to support/resistance is better)
    dist_low = row.get("Dist_Recent_Low", 0)
    dist_high = row.get("Dist_Recent_High", 0)
    if action > 0 and dist_low < 0.08:      # near recent low for longs
        score += weights['location']
    elif action < 0 and dist_high > -0.08:  # near recent high for shorts
        score += weights['location']

    # Optional: Normalize or cap the final score
    score = max(min(score, 100), -50)  # example bounds

    return score

#######################################################################
# Helper files for Episodic Pivot System
#######################################################################
# detect regime
def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

# True base detection after trend
def detect_base_with_context(price):

    # --- Step 1: Identify prior run-up ---
    close = price["Close"]

    if len(close) < 150:
        return False, None, None

    # Use last 3–6 months to find peak
    lookback = close.iloc[-126:]  # 6 months

    peak_idx = lookback.idxmax()
    peak_price = lookback.max()

    # Ensure peak is not too recent (otherwise no base yet)
    if peak_idx == close.index[-1]:
        return False, None, None

    # --- Step 2: Define base AFTER peak ---
    base = price.loc[peak_idx:]

    if len(base) < 15:
        return False, None, None

    base_high = base["High"].max()
    base_low = base["Low"].min()

    # --- Step 3: Pullback constraint (<= 25%) ---
    drawdown = (peak_price - base_low) / peak_price

    if drawdown > 0.25:
        return False, base_high, base_low

    # --- Step 4: Compression (tight + quiet) ---
    base_range = (base_high - base_low) / base_high
    tightness = base["Close"].std() / base["Close"].mean()

    if base_range < 0.25 and tightness < 0.08:
        return True, base_high, base_low

    return False, base_high, base_low

# earnings event detection
def detect_event(price):

    gap = (
        price["Open"].iloc[-1] -
        price["Close"].iloc[-2]
    ) / price["Close"].iloc[-2]

    vol_ratio = (
        price["Volume"].iloc[-1] /
        price["Volume"].iloc[-20:].mean()
    )

    if abs(gap) > 0.03 and vol_ratio > 1.5:
        return True, gap, vol_ratio # Return all 3 values when True

    return False, gap, vol_ratio # Return all 3 values even when False

def max_runup_valid(series):
    series = series.dropna()
    if len(series) < 20:
        return 0

    cum_min = series.cummin()
    runup = series / cum_min - 1
    return runup.max()

def score_ep(price):

    trend = price["Close"].pct_change(126).iloc[-1]

    volume = price["Volume"].iloc[-1] / price["Volume"].iloc[-20:].mean()

    volatility = price["Close"].pct_change().std()

    score = (
        trend * 40 +
        volume * 30 +
        (1 / volatility) * 30
    )

    return score

# Advanced EP scanner
def advanced_ep_scan(df,lookback_days=60):

    signals = []

    # Get the actual unique tickers present in the first level of the MultiIndex
    actual_tickers = df.columns.get_level_values(0).unique()

    for ticker in actual_tickers:

        price = df[ticker].dropna()

        if len(price) < 150:
            continue

        # check last N days
        for i in range(lookback_days, 0, -1):
             hist = price.iloc[:-i]
             ma10 = hist["Close"].rolling(10).mean()
             ma20 = hist["Close"].rolling(20).mean()
             ma50 = hist["Close"].rolling(50).mean()

             alignment = (ma10 > ma20) & (ma20 > ma50)

             price_valid = hist["Close"].copy()
             price_valid[~alignment] = np.nan

             trend_3m = max_runup_valid(price_valid.iloc[-63:])
             trend_6m = max_runup_valid(price_valid.iloc[-126:])

             if trend_3m < 0.30 and trend_6m < 0.30:
                continue


             # Base detection WITH trend context
             base_ok, base_high, base_low = detect_base_with_context(hist)

             if not base_ok:
                continue

            # Event detection
             event_ok, gap, vol_ratio = detect_event(hist)

             if not event_ok:
                continue

             entry_price = hist["Close"].iloc[-1]
             stop_price = hist["Low"].iloc[-1]
             regime = hist["regime"].iloc[-1]

             signals.append({
                "Ticker": ticker,
                "Signal_Date": hist.index[-1],
                "Entry": entry_price,
                "Stop": stop_price,
                "Gap_%": round(gap * 100, 2),
                "Volume_Ratio": round(vol_ratio, 2),
                "Trend_3M_%": round(trend_3m * 100, 1),
                "Trend_6M_%": round(trend_6m * 100, 1),
                "BaseHigh": base_high,
                "BaseLow": base_low,
                "regime": regime,
                "Score": score_ep(hist)
            })

    signals = pd.DataFrame(signals)

    if len(signals) == 0:
        # print("No EP events in the last", lookback_days, "days")
        return signals

    signals = signals.sort_values("Signal_Date", ascending=False)

    signals['Score'] = [score_ep(df[ticker]) for ticker in signals['Ticker']]

    return signals

# download logistic regression model
#######################################################################
from google.colab import drive
drive.mount('/content/drive')

import joblib
# download the logistic model from google drive
pipeline = joblib.load('/content/drive/MyDrive/ML/log_pipeline.pkl')

# download the model_long and model_short from google drive
model_long = joblib.load('/content/drive/MyDrive/ML/model_long.pkl')
model_short = joblib.load('/content/drive/MyDrive/ML/model_short.pkl')

#######################################################################
# Helper files for Momentum System
#######################################################################

# make a function to calculate EMAs
def calculate_ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def compute_heikin_ashi(data):
    """
    Compute Heikin-Ashi candles for a MultiIndex OHLC DataFrame.

    Parameters
    ----------
    data : pd.DataFrame
        MultiIndex columns: (ticker, field)
        Required fields: Open, High, Low, Close

    Returns
    -------
    ha_candles : pd.DataFrame
        MultiIndex columns: (ticker, HA_open, HA_high, HA_low, HA_close)
    """

    ha_candles_dict = {}

    tickers = data.columns.get_level_values(0).unique()

    for ticker in tqdm(tickers, desc="Calculating Heikin Ashi"):

        if ticker not in data.columns.get_level_values(0):
            continue

        ticker_data = data[ticker].copy()

        required_cols = ['Open', 'High', 'Low', 'Close']
        if not all(col in ticker_data.columns for col in required_cols):
            print(f"Skipping {ticker}, missing OHLC")
            continue

        o = ticker_data['Open']
        h = ticker_data['High']
        l = ticker_data['Low']
        c = ticker_data['Close']


        # --- HA Close (vectorized) ---
        ha_close = (o + h + l + c) / 4

        # --- HA Open (recursive) ---
        ha_open = np.zeros(len(ha_close))
        ha_open[0] = (o.iloc[0] + c.iloc[0]) / 2  # initialization

        for i in range(1, len(ha_close)):
            ha_open[i] = (ha_open[i-1] + ha_close.iloc[i-1]) / 2

        ha_open = pd.Series(ha_open, index=ha_close.index)

        # --- HA High / Low ---
        ha_high = pd.concat([h, ha_open, ha_close], axis=1).max(axis=1)
        ha_low  = pd.concat([l, ha_open, ha_close], axis=1).min(axis=1)

        df_ha = pd.DataFrame({
            'HA_open': ha_open,
            'HA_high': ha_high,
            'HA_low': ha_low,
            'HA_close': ha_close
            })

        ha_candles_dict[ticker] = df_ha

    # --- Combine into MultiIndex ---
    all_ha_data_list = []

    for ticker, df_ha in ha_candles_dict.items():
        df_ha.columns = pd.MultiIndex.from_product([[ticker], df_ha.columns])
        all_ha_data_list.append(df_ha)

    if all_ha_data_list:
        ha_candles = pd.concat(all_ha_data_list, axis=1)
        ha_candles = ha_candles.sort_index(axis=1)
    else:
        ha_candles = pd.DataFrame()

    print("Heikin Ashi candles calculated successfully.")

    return ha_candles

# Calculate stoch RSI
def calculate_stoch_rsi(series, rsi_period=14, stoch_k_period=14, stoch_d_period=3):
    """
    Calculates StochRSI %K and %D from a price series (e.g., Close prices).

    Args:
        series (pd.Series): Close prices
        rsi_period, stoch_k_period, stoch_d_period: parameters

    Returns:
        StochRSI_K (pd.Series), StochRSI_D (pd.Series)
    """
    delta = series.diff()

    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)

    avg_gain = gains.ewm(alpha=1/rsi_period, adjust=False).mean()
    avg_loss = losses.ewm(alpha=1/rsi_period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    rsi_low = rsi.rolling(window=stoch_k_period).min()
    rsi_high = rsi.rolling(window=stoch_k_period).max()

    stochrsi_k = (rsi - rsi_low) / (rsi_high - rsi_low) * 100
    stochrsi_d = stochrsi_k.rolling(window=stoch_d_period).mean()

    return stochrsi_k, stochrsi_d


def momentum_action_signals(df):
    """
    Heikin-Ashi Momentum Entry Signals (WITH VOLUME, NO EXIT)

    Logic:
    - HA momentum (multi-candle strength)
    - EMA trend filter
    - Momentum confirmation (StochRSI mid-zone)
    - Breakout continuation
    - Volume expansion + surge

    Output:
    buy: 1 = long, -1 = short
    """

    df = df.copy()
    df = df.sort_index(axis=1)

    signals = [] # Initialize signals as an empty list

    for ticker in df.columns.levels[0]:

        required = [
            'HA_open','HA_high','HA_low','HA_close',
            'EMA20', 'EMA50','EMA200', 'StochRSI_K','StochRSI_D',
            'Volume','Open', 'regime'
        ]

        if not all((ticker, col) in df.columns for col in required):
            print(f"Skipping {ticker}, missing columns")
            continue

        o = df[(ticker,'HA_open')]
        h = df[(ticker,'HA_high')]
        l = df[(ticker,'HA_low')]
        c = df[(ticker,'HA_close')]

        ema20 = df[(ticker,'EMA20')]
        ema50 = df[(ticker,'EMA50')]
        ema200 = df[(ticker,'EMA200')]

        k = df[(ticker,'StochRSI_K')]
        d = df[(ticker,'StochRSI_D')]

        vol = df[(ticker,'Volume')]
        raw_open = df[(ticker,'Open')]

        # --- Volume features ---
        vol_ma20 = vol.rolling(20).mean()
        vol_ma50 = vol.rolling(50).mean()

        df[(ticker,'buy')] = 0

        for i in range(5, len(df)):  # need more history now

            # --- Current candle ---
            op = o.iloc[i]
            hi = h.iloc[i]
            lo = l.iloc[i]
            cl = c.iloc[i]

            # --- Previous candles ---
            c1 = c.iloc[i-1]
            c2 = c.iloc[i-2]

            o1 = o.iloc[i-1]
            o2 = o.iloc[i-2]

            l1 = l.iloc[i-1]
            h1 = h.iloc[i-1]

            l2 = l.iloc[i-2]
            h2 = h.iloc[i-2]

            # --- Indicators ---
            e20 = ema20.iloc[i]
            e50 = ema50.iloc[i]
            e200 = ema200.iloc[i]

            kk = k.iloc[i]
            dd = d.iloc[i]

            # --- Volume ---
            v = vol.iloc[i]
            v1 = vol.iloc[i-1]
            v2 = vol.iloc[i-2]

            v_ma20 = vol_ma20.iloc[i]
            v_ma50 = vol_ma50.iloc[i]

            buy_signal = 0

            regime = df[(ticker, 'regime')].iloc[i]

            # =====================================================
            # LONG: HA Momentum + Volume Confirmation
            # =====================================================
            if (
                # --- Trend alignment ---
                cl > e20 > e50 > e200 and

                # --- HA bullish momentum ---
                cl > op and
                c1 > o1 and
                c2 > o2 and

                # --- Strong candles (no lower wick) ---
                lo >= min(op, cl) and
                l1 >= min(o1, c1) and

                # --- Higher highs ---
                hi > h1 > h2 and

                # --- Momentum confirmation ---
                kk > dd and kk > 50 and

                # ================= VOLUME =================
                # Expansion
                v > v_ma20 and

                # Surge (institutional participation)
                v > 1.5 * v_ma50 and

                # Rising volume (optional but powerful)
                v > v1 > v2
            ):
                buy_signal = 1
                entry_price = c.iloc[i]
                stop_price = entry_price * (1 - 0.05)

                signals.append({
                    "Ticker": ticker,
                    "Signal_Date": df.index[i],
                    "Entry": entry_price,
                    "Stop": stop_price,
                    "Buy_Signal": buy_signal,
                    "EMA20": e20,
                    "EMA50": e50,
                    "EMA200": e200,
                    "StochRSI_K": kk,
                    "StochRSI_D": dd,
                    "Volume20": v_ma20,
                    "Volume50": v_ma50,
                    "Volume": vol.iloc[i],
                    "regime": regime,
                })

            # =====================================================
            # SHORT: HA Momentum + Volume Confirmation
            # =====================================================
            elif (
                cl < e20 < e50 < e200 and

                cl < op and
                c1 < o1 and
                c2 < o2 and

                hi <= max(op, cl) and
                h1 <= max(o1, c1) and

                lo < l1 < l2 and

                kk < dd and kk < 50 and

                # ================= VOLUME =================
                v > v_ma20 and
                v > 1.5 * v_ma50 and
                v > v1 > v2
            ):
                buy_signal = -1
                entry_price = c.iloc[i]
                stop_price = entry_price * (1 + 0.05)

                signals.append({
                    "Ticker": ticker,
                    "Signal_Date": df.index[i],
                    "Entry": entry_price,
                    "Stop": stop_price,
                    "Buy_Signal": buy_signal,
                    "EMA20": e20,
                    "EMA50": e50,
                    "EMA200": e200,
                    "StochRSI_K": kk,
                    "StochRSI_D": dd,
                    "Volume20": v_ma20,
                    "Volume50": v_ma50,
                    "Volume": vol.iloc[i],
                    "regime": regime,
                })


    signals_df = pd.DataFrame(signals) # Rename to signals_df to avoid conflict

    if len(signals_df) == 0:
        # print("No EP events in the last", lookback_days, "days")
        return pd.DataFrame() # Return an empty DataFrame if no signals

    signals_df = signals_df.sort_values("Signal_Date", ascending=False)

    return signals_df # Return the DataFrame of signals

# End of Helper files


######################################################################
# Set parameters and download data
######################################################################

# set parameters
START_DATE_1y = (datetime.today() - timedelta(days=365*1)).strftime('%Y-%m-%d') # 1y histroy data
END_DATE = datetime.today().strftime('%Y-%m-%d')
RISK = 0.06 # Engulfing bars risk
EP_RISK = 0.04 # Episodic Pivot risk
M_RISK = 0.051 # Momentum risk

# Define probability thresholds (from previous optimal findings)
long_threshold = 0.5
short_threshold = 0.6

# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS_1y = 200 # Approximately 1 year of trading days

# Fetch tickers and prices
sp500_tickers = get_sp500_tickers()
universe = [t for t in sp500_tickers]

prices_1y = download_stock_prices(universe, start=START_DATE_1y, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in sp500_tickers if prices_1y.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS_1y]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS_1y} days of data.")

spx_1y = yf.download("^GSPC", start=START_DATE_1y, end=END_DATE)
spx_1y.columns = spx_1y.columns.droplevel(1)
spx_1y = label_regime(spx_1y)

data_1y = prices_1y.copy()
aligned_regime = spx_1y['regime'].reindex(data_1y.index, method='ffill')

for ticker in data_1y.columns.get_level_values(0).unique():
    data_1y[ticker, 'regime'] = aligned_regime

stock_last_day = data_1y.index[-1:][0].strftime('%Y-%m-%d')
print(f"The last date of the stock price is {stock_last_day}\n")

######################################################################
# Engulfing Bars Scanner
######################################################################
print("\nEngulfing Bar Signals Scanning...")
signals_1y = screen_engulfing_signals(data_1y)

if signals_1y.empty:
    print("No Engulfing Bars signals were found. Skipping further processing for Engulfing Bars.")
    final_signals = pd.DataFrame() # Initialize final_signals as empty to avoid NameError later
else:
    signals_1y["Signal_Score"] = signals_1y.apply(compute_signal_score, axis=1)
    df = signals_1y.copy()

    # replace Action with 2 features (So one-hot only Direction, keep Strength numeric)
    df["Direction"] = np.sign(df["Action"])
    df["Strength"] = abs(df["Action"])

    # 1. convert 'Action' and 'regime' to One-Hot
    df_encoded = pd.get_dummies(
        df,
        columns=['Direction', 'regime'],
        prefix=['Direction', 'regime'],
        drop_first=True)

    # split dataset
    df_long = df_encoded[df["Action"] > 0].copy()
    df_short = df_encoded[df["Action"] < 0].copy()

    # 2. rebuild feature set
    drop_cols = ['Ticker', 'Signal_Date', 'Action', 'Entry', 'Stop']
    X_long = df_long.drop(columns=drop_cols)

    X_short = df_short.drop(columns=drop_cols)

    # 3. Preprocessing and predict

    # Initialize probabilities Series for all signals_1y with NaN
    all_probabilities = pd.Series(np.nan, index=df_encoded.index)

    # Predict for long signals if X_long_pred is not empty
    if not X_long.empty:
        probabilities_long = model_long.predict_proba(X_long)[:, 1]
        all_probabilities.loc[df_long.index] = probabilities_long

    # Predict for short signals if X_short_pred is not empty
    if not X_short.empty:
        probabilities_short = model_short.predict_proba(X_short)[:, 1]
        all_probabilities.loc[df_short.index] = probabilities_short

    # print("First 5 probabilities from long model:")
    # display(all_probabilities[df_long.index].head())
    # print("First 5 probabilities from short model:")
    # display(all_probabilities[df_short.index].head())

    # Initialize 'gain' in df to 0
    df['gain'] = 0

    # Identify indices for which we have predictions (non-NaN probabilities)
    valid_signal_indices = all_probabilities.dropna().index

    if not valid_signal_indices.empty:
        # Get the original 'Action' values for these valid signals to determine which threshold to use
        actions_for_valid_signals = df.loc[valid_signal_indices, 'Action']

        # Determine the correct threshold for each valid signal (long_threshold for Action > 0, short_threshold otherwise)
        thresholds_for_signals = np.where(
            actions_for_valid_signals > 0, # Condition: If original Action was positive (long)
            long_threshold,                # Result if true: use long_threshold
            short_threshold                # Result if false: use short_threshold (for short/neutral actions)
        )

        # Apply the np.where condition to set 'gain' for the valid signals
        df.loc[valid_signal_indices, 'gain'] = np.where(
            all_probabilities.loc[valid_signal_indices] > thresholds_for_signals,
            1, # If predicted probability is greater than the assigned threshold, it's a gain
            0  # Otherwise, it's not a gain
        )

    # print("First 5 rows of df with the new 'gain' column:")
    # display(df[['Ticker','Signal_Date', 'Action', 'Signal_Score', 'gain']].head())

    last_3days = (datetime.now() - timedelta(days=4)).date()
    last_day = (datetime.now() - timedelta(days=2)).date()

    if datetime.today().weekday() == 1:
        final_signals  = df[(df['gain']==1) & (df['Signal_Date'].dt.date > last_3days)].sort_values('Signal_Score', ascending=False)
    else:
        final_signals  = df[(df['gain']==1) & (df['Signal_Date'].dt.date > last_day)].sort_values('Signal_Score', ascending=False)

    if final_signals.empty:
        print("No Engulfing Bars signals in the last days.")
    else:
        final_signals['Loss'] = final_signals['Entry'] * RISK

        final_signals["Stop_Loss"] = np.where(final_signals['Action'] > 0,
                                            final_signals['Entry'] * (1 - RISK),
                                            final_signals['Entry'] * (1 + RISK))
        final_signals = final_signals[['Ticker', 'Signal_Date', 'Action', 'Entry','Stop_Loss','regime','Signal_Score',
            'Direction', 'Strength', 'gain']]

        if datetime.today().weekday() == 1:
            print("Predicted true Engulfing_Bars signals in the last THREE days, sorted by 'Signal_Score':")
            display(final_signals.sort_values('Signal_Score', ascending=False))
        else:
            print("Predicted true Engulfing_Bars signals in the last day, sorted by 'Signal_Score':")
            display(final_signals.sort_values('Signal_Score', ascending=False))



#####################################################################################
# Episodic Pivot Scanner
#####################################################################################
print("\nEpisodic Pivot Signals Scanning...")
EP_signals_1y = advanced_ep_scan(data_1y, 60)

# Step 2. get the features and scale them. use LightGBM model to predict whether they are profitable
####################################################################################################

EP_df = EP_signals_1y.copy()

# 1. Feature extract
EP_df["BaseDepth"] = (EP_df["BaseHigh"] - EP_df["BaseLow"]) / EP_df["BaseHigh"]
EP_df["Gap_Strength"] = EP_df["Gap_%"] * EP_df["Volume_Ratio"]
EP_df["BasePosition"] = EP_df["Entry"] / EP_df["BaseHigh"]
EP_df["Vol_Expansion"] = EP_df["Volume_Ratio"] * EP_df["Gap_%"]
EP_df["Trend_Regime"] = EP_df["Trend_3M_%"] * EP_df["regime"]
EP_df["Volume_Regime"] = EP_df["Volume_Ratio"] * EP_df["regime"]

# 2. Features / Target
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    'Gap_%', 'Volume_Ratio',
    'Trend_3M_%', 'Trend_6M_%',
    'BaseHigh', 'BaseLow',
    'BaseDepth', 'Gap_Strength',
    'BasePosition',
    'Vol_Expansion',
    'Score',
    'Trend_Regime',
    'Volume_Regime'
]

categorical_features = ['regime']

# 3. Preprocessing and predict

X = EP_df[numeric_features + categorical_features]

probabilities = pipeline.predict_proba(X)[:, 1]

EP_signals_1y['gain'] = np.where(probabilities > 0.6, 1, 0)

# Step 3. Print latest predicted signals
########################################

print(f"\nTOP EP signals\n: {EP_signals_1y.sort_values(by='Signal_Date', ascending=False).head()}")
print('#'*80)
last_3days = (datetime.now() - timedelta(days=3)).date()
EP_signals = EP_signals_1y[EP_signals_1y['Signal_Date'].dt.date > last_3days]

if EP_signals.empty:
    print("\nNo EP signals in last THREE days.")
else:
    print(f"Latest's EP signals: {EP_signals}")
    print(f"Latest's STRONG EP signals: {EP_signals[EP_signals['gain'] == 1]}")
    print(f"Pay attention to the Risk, Risk cut is {EP_RISK}")

#####################################################################################
# Momentum Signals Scanner
#####################################################################################
print("\nMomentum Signals Scanning...\n")
# data_1y
# Calculate Heikin-Ashi candles
ha_data = compute_heikin_ashi(data_1y)
ha_df = pd.concat([data_1y, ha_data], axis=1)
ha_df = ha_df.sort_index(axis=1)
# Calculate StochRSI
for ticker in tqdm(ha_df.columns.levels[0], desc="Calculating StochRSI"):
  close_series = ha_df[(ticker, 'Close')].copy()

  if close_series.isna().all():
      print(f"Skipping {ticker}: no close data")
      continue

  try:
        k_line, d_line = calculate_stoch_rsi(close_series)

        # Assign correctly — one column at a time
        ha_df[(ticker, 'StochRSI_K')] = k_line
        ha_df[(ticker, 'StochRSI_D')] = d_line
        # ha_data[(ticker, 'EMA200')] = data[(ticker, 'Close')].ewm(span=200, adjust=False).mean()

  except Exception as e:
        print(f"Error calculating StochRSI for {ticker}: {e}")

# prepare S&P 500 index data
# spx = yf.download("^GSPC", start=START_DATE, end=END_DATE)
# spx.columns = spx.columns.droplevel(1)
# spx = label_regime(spx)
# # combine data with spx['regime']
# # First, align the spx['regime'] Series to the 'data' DataFrame's index.
# # Using 'ffill' will propagate the last valid observation forward to fill any missing dates.
# aligned_regime = spx['regime'].reindex(df.index, method='ffill')

# # Iterate through each unique ticker and add the aligned 'regime' series
# # as a new sub-column under that ticker in the MultiIndex DataFrame.
# for ticker in df.columns.get_level_values(0).unique():
#     df[ticker, 'regime'] = aligned_regime

for ticker in tqdm(ha_df.columns.levels[0], desc="Calculating EMA"):
  close_series = ha_df[(ticker, 'Close')].copy().squeeze() # Add .squeeze() here to ensure it's a Series
  if close_series.isna().all():
      print(f"Skipping {ticker}: no close data")
      continue
  try:
    ha_df[(ticker, 'EMA20')] = calculate_ema(close_series, 20)
    ha_df[(ticker, 'EMA50')] = calculate_ema(close_series, 50)
    ha_df[(ticker, 'EMA200')] = calculate_ema(close_series, 200)
  except Exception as e:
    print(f"Error calculating EMA for {ticker}: {e}")

# Scan the signals
momentum_signals = momentum_action_signals(ha_df)

# make final report
cols = ['Ticker', 'Signal_Date', 'Entry', 'Buy_Signal', 'regime']
# last_3days = (datetime.now() - timedelta(days=3)).date()
final_signals = momentum_signals[momentum_signals['Signal_Date'].dt.date > last_3days][cols]

if not final_signals.empty:
  final_signals['Strength'] = np.where(final_signals['Buy_Signal'] == final_signals['regime'], 1, 0)
  final_signals['Stop_Loss'] = np.where(
      final_signals['Buy_Signal'] > 0,
      final_signals['Entry'] * (1 - RISK),
      final_signals['Entry'] * (1 + RISK)
  )
  print("****************************************************************\n")
  print(f"Found {len(final_signals)} Momentum Signals in the last 3 days!\n")
  print(final_signals.sort_values(['Signal_Date','Strength'], ascending=False))
  print(f"Pay attention to the Risk, Risk cut is {M_RISK}")
else:
  print("****************************************************************\n")
  print(f"Sorry!")
  print(f"There is no Momentum Signals found recently!")
  print(f"Keep looking tomorrow!")

In [ ]:
signals_1y.sort_values(["Signal_Date","Signal_Score"], ascending = False).head(15)